In [51]:
import os
import pandas
from utils import partial_match_scores

dataset_root = "/home/xzhao/workspace/GYB_self-ensemble/datasets"
ds_name = "myriadlama"
model_name = "llama3.2_3b_it"
# model_name = "qwen2.5_7b_it"

single_para_qapair = False

def get_filenames(
        modifyattn, modifyrope, scale_score,
        repeat_paras, single_para_qapair, 
        num_paraphrases, num_samples=5):
    dump_file = f"{dataset_root}/{ds_name}/{model_name}/myriadlama."
    if modifyattn:
        dump_file += "modifyattn."
    if modifyrope:
        dump_file += "modifyrope."
    if repeat_paras:
        dump_file += "repeatparas."
    if scale_score:
        dump_file += "scalescore20."
    
    if single_para_qapair:
        dump_file += "singleparaqapair."


    dump_file += f"{num_samples}samples.{num_paraphrases}paras.feather"
    # print(f"Loading from {dump_file}")
    return dump_file

def _calculate_accuracy(df, label):
    predicts = [[pred] for pred in df["predict_lemma"].tolist()]
    answers = [answers for answers in df["answer_lemmas"]]
    acc = partial_match_scores(predicts, answers, birdirect=True)    
    print(f"{label} Accuracy: {acc:.4f}")

def calculate_accuracy(
        modifyattn, modifyrope, scale_score,
        repeat_paras, num_paraphrases):
    filename = get_filenames(modifyattn, modifyrope, scale_score, repeat_paras, False, num_paraphrases, num_samples=5)
    if os.path.exists(filename) is False:
        print(f"File {filename} does not exist!")
        return None
    df = pandas.read_feather(filename)
    label = f"MyriadLlama {'+Attn' if modifyattn else ''} {'+Rope' if modifyrope else ''} {'+RepeatParas' if repeat_paras else ''} {num_paraphrases} Paras"
    _calculate_accuracy(df, label)
    return df

In [52]:
baseline_fn = f"{dataset_root}/{ds_name}/{model_name}/baseline_origin.feather"
baseline2_fn = f"{dataset_root}/{ds_name}/{model_name}/baseline_per_prompt.feather"


baseline = pandas.read_feather(baseline2_fn)
baseline["predict_lemma"] = baseline["predict_lemma"].apply(lambda xs: xs[0])
_calculate_accuracy(baseline, "MyriadLlama Baseline Per Prompt")

MyriadLlama Baseline Per Prompt Accuracy: 0.4105


In [53]:
def report_accuracy(modifyattn, modifyrope, scale_score, repeat_paras):
    para2 = calculate_accuracy(modifyattn=modifyattn, modifyrope=modifyrope, scale_score=scale_score, repeat_paras=repeat_paras, num_paraphrases=2)
    para3 = calculate_accuracy(modifyattn=modifyattn, modifyrope=modifyrope, scale_score=scale_score, repeat_paras=repeat_paras, num_paraphrases=3)
    para4 = calculate_accuracy(modifyattn=modifyattn, modifyrope=modifyrope, scale_score=scale_score, repeat_paras=repeat_paras, num_paraphrases=4)
    para5 = calculate_accuracy(modifyattn=modifyattn, modifyrope=modifyrope, scale_score=scale_score, repeat_paras=repeat_paras, num_paraphrases=5)
    # return para2, para3, para4, para5

In [54]:
print("=== No attention or rope modifications, single QA section ===")
modifyattn, modifyrope, repeat_paras = False, False, False
print("--- Without score scaling ---")
report_accuracy(modifyattn, modifyrope, scale_score=False, repeat_paras=repeat_paras)
print("--- With score scaling ---")
report_accuracy(modifyattn, modifyrope, scale_score=True, repeat_paras=repeat_paras)
print()

print("=== With only attention modifications, single QA section ===")
modifyattn, modifyrope, repeat_paras = True, False, False
print("--- Without score scaling ---")
report_accuracy(modifyattn, modifyrope, scale_score=False, repeat_paras=repeat_paras)
print("--- With score scaling ---")
report_accuracy(modifyattn, modifyrope, scale_score=True, repeat_paras=repeat_paras)
print()


print("=== With attention and rope modifications, single QA section ===")
modifyattn, modifyrope, repeat_paras = True, True, False
print("--- Without score scaling ---")
report_accuracy(modifyattn, modifyrope, scale_score=False, repeat_paras=repeat_paras)
print("--- With score scaling ---")
report_accuracy(modifyattn, modifyrope, scale_score=True, repeat_paras=repeat_paras)
print()

=== No attention or rope modifications, single QA section ===
--- Without score scaling ---
MyriadLlama    2 Paras Accuracy: 0.4141
MyriadLlama    3 Paras Accuracy: 0.0480
MyriadLlama    4 Paras Accuracy: 0.0061
MyriadLlama    5 Paras Accuracy: 0.0010
--- With score scaling ---
MyriadLlama    2 Paras Accuracy: 0.0222
MyriadLlama    3 Paras Accuracy: 0.0009
MyriadLlama    4 Paras Accuracy: 0.0001
MyriadLlama    5 Paras Accuracy: 0.0000

=== With only attention modifications, single QA section ===
--- Without score scaling ---
MyriadLlama +Attn   2 Paras Accuracy: 0.4038
MyriadLlama +Attn   3 Paras Accuracy: 0.4149
MyriadLlama +Attn   4 Paras Accuracy: 0.4169
MyriadLlama +Attn   5 Paras Accuracy: 0.4200
--- With score scaling ---
MyriadLlama +Attn   2 Paras Accuracy: 0.4036
MyriadLlama +Attn   3 Paras Accuracy: 0.4149
MyriadLlama +Attn   4 Paras Accuracy: 0.4168
MyriadLlama +Attn   5 Paras Accuracy: 0.4200

=== With attention and rope modifications, single QA section ===
--- Without scor

In [44]:
print("=== No rope modifications or score scaling, single QA section ===")
scale_score, modifyrope, repeat_paras = False, False, False
print("--- Without attention modifications ---")
report_accuracy(False, modifyrope=modifyrope, scale_score=scale_score, repeat_paras=repeat_paras)
print("--- With attention modifications ---")
report_accuracy(True, modifyrope=modifyrope, scale_score=scale_score, repeat_paras=repeat_paras)

=== No rope modifications or score scaling, single QA section ===
--- Without attention modifications ---
MyriadLlama    2 Paras Accuracy: 0.4141
MyriadLlama    3 Paras Accuracy: 0.0480
MyriadLlama    4 Paras Accuracy: 0.0061
MyriadLlama    5 Paras Accuracy: 0.0010
--- With attention modifications ---
MyriadLlama +Attn   2 Paras Accuracy: 0.4038
MyriadLlama +Attn   3 Paras Accuracy: 0.4149
MyriadLlama +Attn   4 Paras Accuracy: 0.4169
MyriadLlama +Attn   5 Paras Accuracy: 0.4200


In [45]:
print("=== With attention modifications but no score scaling, single QA section ===")
modifyattn, scale_score, repeat_paras = True, False, False
print("--- Without rope modifications ---")
report_accuracy(modifyattn=modifyattn, modifyrope=False, scale_score=scale_score, repeat_paras=repeat_paras)
print("--- With rope modifications ---")
report_accuracy(modifyattn=modifyattn, modifyrope=True, scale_score=scale_score, repeat_paras=repeat_paras)

=== With attention modifications but no score scaling, single QA section ===
--- Without rope modifications ---
MyriadLlama +Attn   2 Paras Accuracy: 0.4038
MyriadLlama +Attn   3 Paras Accuracy: 0.4149
MyriadLlama +Attn   4 Paras Accuracy: 0.4169
MyriadLlama +Attn   5 Paras Accuracy: 0.4200
--- With rope modifications ---
MyriadLlama +Attn +Rope  2 Paras Accuracy: 0.4026
MyriadLlama +Attn +Rope  3 Paras Accuracy: 0.4114
MyriadLlama +Attn +Rope  4 Paras Accuracy: 0.4112
MyriadLlama +Attn +Rope  5 Paras Accuracy: 0.4152
